<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Bölüm 3: Alıştırma Çözümleri

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.4
torch version: 2.7.1
tokenizers version: 0.21.4


&nbsp;
## Alıştırma 3.1: Daha fazla test durumu eklemek

- Ekleyebileceğimiz sayısız farklı test durumu var
- Aşağıda ilginç olanlardan bir seçki yer alıyor

In [ ]:
from reasoning_from_scratch.ch03 import (
    run_demos_table
)

more_tests = [
    # Farklı parantez türleri
    ("check_17", "[1, 2]", "(1, 2)", True),

    # Bilimsel gösterim
    ("check_18", "1e-3", "0.001", True),

    # Şapka işaretli üs ile cebirsel sadeleştirme
    ("check_19", "(-3)^2", "9", True),

    # Unicode eksi (U+2212) ile ASCII tire-eksi karşılaştırması
    ("check_20", "−1", "-1", True),

]

run_demos_table(more_tests)

Test     | Expect | Got   | Status
check_17 | True   | True  | PASS  
check_18 | True   | True  | PASS  
check_19 | True   | True  | PASS  
check_20 | True   | False | FAIL  

Passed 3/4


- Görüldüğü gibi, testler `check_20` dışında tüm durumlarda geçiyor; bu test, normal işareti insan gözüyle ayırt edilemeyen bir Unicode eksi işaretiyle değiştiriyor
- Bu test durumunu, `normalize_text` fonksiyonuna aşağıdaki satırlardan birini herhangi bir yere ekleyerek düzeltebiliriz

```python
text = text.replace("−", "-")
# or
text = text.replace("\u2212", "-")
```

- İlk bakışta ilginç görünen bir başka test de şu:

In [2]:
extra_tests_1 = [
    ("check_21", "Text around answer 3.", "3", True)
]

run_demos_table(extra_tests_1)

Test     | Expect | Got   | Status
check_21 | True   | False | FAIL  

Passed 0/1


- Kodumuzun metin içeren bu tür durumları işleyemediği görünse de, bu aslında kötü tasarlanmış bir testtir
- Pratikte `run_demos_table` fonksiyonu özellikle `grade_answer` fonksiyonunu test etmeye yöneliktir; ne fazlası ne eksiği
- `grade_answer` fonksiyonu yanıtın tamamını bu biçimde asla almaz; çünkü yanıt, kendisine verilmeden önce metinden ayıklanmış olurdu

Yani metin yanıtlarını test etmek istiyorsak testi şöyle çağırmamız gerekir:

In [3]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate
)


extra_tests_2 = [
    ("check_21",
     extract_final_candidate("Text around answer 3."),
     "3", True)
]
run_demos_table(extra_tests_2)

Test     | Expect | Got  | Status
check_21 | True   | True | PASS  

Passed 1/1


&nbsp;
## Alıştırma 3.2: Ortalama yanıt uzunluğunu hesaplamak

- A seçeneği: `evaluate_math500_stream` fonksiyonunu aşağıdaki satırları ekleyerek değiştirebiliriz:

```python
# ...
# below `num_correct = 0`
total_len = 0

# ...
# inside for i, row in enumerate(math_data, start=1):
# anywhere below `gen_text = ...`
total_len += len(tokenizer.encode(gen_text))

# ...
# anywhere at the bottom before the return statement
avg_len = total_len / num_examples
print(f"Average length: {avg_len:.2f} tokens")
```

- Alternatif olarak, ana bölümde `evaluate_math500_stream` fonksiyonunu çalıştırdığımızda oluşturulan `.jsonl` dosyalarından da yanıt uzunluklarını hesaplayabiliriz
- Önce `.jsonl` dosyasını şöyle yüklüyoruz:

In [5]:
import json
from pathlib import Path

WHICH_MODEL = "base"

dev_name = "mps"  # e.g., "cuda", "cpu"

# Bu yolu değiştirmeniz gerekebilir:
local_path = Path(f"math500-{dev_name}.jsonl")
if not local_path.exists():
    raise FileNotFoundError(
        f"{local_path} not found. Run ch03_main.ipynb to create it."
    )

results = []
with open(local_path, "r") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

print("Number of entries:", len(results))


Number of entries: 10


- Her kaydın birden çok anahtarı olduğunu, ancak bizi yalnızca modelin tam yanıtını içeren `"generated_text"` anahtarının ilgilendirdiğini unutmayın:

In [6]:
print(results[0].keys())

dict_keys(['index', 'problem', 'gtruth_answer', 'generated_text', 'extracted', 'correct'])


- Her kaydın birden çok anahtarı olduğunu unutmayın; ancak bizi yalnızca modelin tam yanıtını içeren `"generated_text"` anahtarı ilgilendiriyor:

In [7]:
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer
)

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

✓ qwen3/tokenizer-base.json already up-to-date


- Ardından ortalama uzunluğu şöyle hesaplayabiliriz; bu, `evaluate_math500_stream` fonksiyonunu nasıl değiştirebileceğimize benzer:

In [8]:
total_len = 0

for item in results:
    num_tokens = len(tokenizer.encode(item["generated_text"]))
    total_len += num_tokens

avg_len = total_len / len(results)
print(f"Average length: {avg_len:.2f} tokens")

Average length: 98.00 tokens


| Kip       | Cihaz   | Ortalama uzunluk | MATH-500 boyutu |
|-----------|---------|----------------|----------------|
| Temel     | CPU     | 97.3           | 10             |
| Temel     | MPS     | 98.0           | 10             |
| Akıl yür. | CPU     | 891.80         | 10             |
| Akıl yür. | MPS     | 1159.30        | 10             |
|           |         |                |                |
| Temel     | CUDA    | 96.74          | 500            |
| Akıl yür. | CUDA    | 1361.21        | 500            |


- Görüldüğü ve beklendiği gibi, akıl yürütme modeli çok daha uzun yanıtlar yazıyor

&nbsp;
## Alıştırma 3.3: Değerlendirme veri kümesini genişletmek ya da değiştirmek

- Modeli daha büyük bir veri kümesinde değerlendirmek için `math_data[:10]` ifadesini farklı bir dilime ya da daha büyük bir sayıya (en fazla 500) değiştirmemiz yeterli

```python
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, 
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)
```

- Aşağıdaki tablo farklı veri kümesi boyutları için doğruluk değerlerini gösteriyor (MATH-500 test kümesi zaten karıştırılmış olduğu için ek bir karıştırma uygulanmadı)

| Kip       | Cihaz   | Doğruluk | MATH-500 boyutu |
|-----------|---------|----------|----------------|
| Temel     | CUDA    | %30.0    | 10             |
| Temel     | CUDA    | %34.0    | 50             |
| Temel     | CUDA    | %27.0    | 100            |
| Temel     | CUDA    | %31.0    | 200            |
| Temel     | CUDA    | %15.3    | 500            |
|           |         |          |                |
| Akıl yür. | CUDA    | %90.0    | 10             |
| Akıl yür. | CUDA    | %58.0    | 50             |
| Akıl yür. | CUDA    | %58.0    | 100            |
| Akıl yür. | CUDA    | %56.0    | 200            |
| Akıl yür. | CUDA    | %48.2    | 500            |

- Yukarıdaki sonuçlardan görebileceğimiz gibi, ilk 10 örnek, 500 örneğin tamamı üzerinde ölçülen MATH-500 başarımını pek temsil etmiyor

- Ayrıca, MATH-500'e benzer bir tarzda tamamen yeni bir veri kümesi de oluşturabiliriz
- Örneğin, bu depoda MATH-500 tarzında bir veri kümesi bulunuyor; ana bölümde dosya adını `math500_test.json` yerine `math_new50_exercise.json` yaparak kullanabiliriz (bu veri kümesi kitabın GitHub deposunda yer alıyor: https://github.com/rasbt/reasoning-from-scratch/tree/main/ch03/01_main-chapter-code)
- Temel modelin ve akıl yürütme modelinin başarımı şöyledir:
    - temel: %36.0 (18/50)
    - akıl yürütme: %80.0 (40/50)
- Buradan, özgün MATH-500 test veri kümesi Qwen3'ün eğitim veri kümesine dahil edilmiş olsa bile, modelin yeni matematik sorularında benzer başarım gösterdiği sonucunu çıkarabiliriz; bu da özgün MATH-500 verisine kapsamlı bir aşırı öğrenme yaşamadığını gösterir

&nbsp;
## Alıştırma 3.4: Farklı istem şablonlarını denemek 

- Bölümde önerilene benzer alternatif istemi kullanabiliriz; bu istem, "Question" yerine "Problem" kullanacak şekilde değiştirilmiştir:

```python
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template
```

- Bu istemi kullanmak, temel modelin 500 örnek üzerindeki başarımını %15.3'ten %31.2'ye çıkarır
- Bu gözlemlerden, temel modelin istem biçimine akıl yürütme modeline göre çok daha duyarlı olduğu sonucunu çıkarabiliriz (bunun nedeni büyük olasılıkla eğitim kümesinden bazı istem biçimli MATH-500 örneklerini ezberlemiş olmasıdır); ikincisi ise büyük ölçüde etkilenmemiş görünüyor